In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
import torch
import torch.nn.functional as F


class ConfSorter(ABC):

    @abstractmethod
    def sort(self):
        pass
    # end
# end

class TruthSorter(ConfSorter):
    pass
# end

class MaxSorter(ConfSorter):
    pass
# end

class RandomSorter(ConfSorter):
    pass
# end

sorter = TruthSorter()
sorter = MaxSorter()
sorter = RandomSorter()

class LlaDAModel():
    pass
# end



class DiffusionQuotaHelper(ABC):
    @abstractmethod
    def get_quota(self, step_current):
        pass
    # end
# end

class BlockDiffusionQuotaHelper(DiffusionQuotaHelper):
    def __init__(self, block_mask_index: torch.Tensor, steps_per_block: int) -> torch.Tensor:
        device = block_mask_index.device
        dtype = torch.long

        total = block_mask_index.sum(dim=1)                  # (B,)
        base  = torch.div(total, steps_per_block, rounding_mode='floor')  # (B,)
        rem   = total - base * steps_per_block                         # (B,)

        # Start with base for all steps
        num_transfer_tokens = base.unsqueeze(1).expand(-1, steps_per_block).to(dtype)  # (B, steps)

        # Add +1 to the first `rem[b]` steps for each batch b — without tensor slicing
        cols = torch.arange(steps_per_block, device=device).unsqueeze(0)               # (1, steps)
        add_mask = cols < rem.unsqueeze(1)                                   # (B, steps)
        self.num_transfer_tokens = num_transfer_tokens + add_mask.to(dtype)       # (B, steps)
    # end

    def get_quota(self, step_current):
        quota_current = self.num_transfer_tokens[:, step_current]
        return quota_current
    # end
# end


class SimpleLogitsSnapshot(ABC):
    def __init__(self, logits, x, y):
        self.logits = logits
        self.x = x
        self.y = y
    # end

    def transform_logits(self):
        logits = self.logits
        x = self.x
        y = self.y
        id_mask = self.id_mask

        mask_mask = (x == id_mask)

        x0 = torch.argmax(logits, dim=-1)
        p = F.softmax(logits.to(torch.float64), dim=-1)

        x0_p = self.gather_x0_p(p, x0, y)
        neg_inf = torch.tensor(torch.finfo(p.dtype).min, device=p.device, dtype=p.dtype)    # TODO: might be error here
        conf = torch.where(mask_mask, x0_p, neg_inf)

        self.conf = conf
        return self
    # end

    @abstractmethod
    def gather_x0_p(p, x0, y):
        pass
    # end

    def set_mask_id(self, id_mask):
        self.id_mask = id_mask
        return self
    # end

    def generate_sorted_sequence(self, sorter):
        mask_mask = (self.x == self.id_mask)

        idx_sorted = sorter.sort(self.conf, mask_mask)
        return idx_sorted
    # end

    def generate_transfer_index(self, idx_sorted, quota, sorter):
        mask_mask = (self.x == self.id_mask)
        confidence = self.conf

        B, L = confidence.shape
        # Build a mask that is True for the first k[b] columns in each row (sorted order)
        cols = torch.arange(L, device=confidence.device).unsqueeze(0).expand(B, L)   # (B, L)
        k_expanded = quota.unsqueeze(1).expand(B, L)                   # (B, L)
        select_sorted = cols < k_expanded                                            # (B, L) bool for top k

        # Scatter the sorted True/False back to original column order
        # Use integer scatter then cast to bool (scatter_ on bool can be finicky across versions)
        transfer_int = torch.zeros(B, L, device=confidence.device, dtype=torch.int8) # (B, L)
        transfer_int = transfer_int.scatter(1, idx_sorted, select_sorted.to(torch.int8))
        transfer_index = transfer_int.bool() & mask_mask  # ensure we never select unmasked        
        return transfer_index
    # end

# end


class SimpleLogitsTruthSnapshot(SimpleLogitsSnapshot):
    def gather_x0_p(self, p, x0, y):
        return torch.gather(p, dim=-1, index=y.unsqueeze(-1))
    # end
# end

class SimpleLogitsConfSnapshot(SimpleLogitsSnapshot):
    def gather_x0(self, p, x0, y):
        return torch.gather(p, dim=-1, index=x0)
    # end
# end













@dataclass
class DiffusionConfig:
    num_blocks: int
# end


model = LlaDAModel()
model.enable_past_kv()

# 处理past_key_values
def run_model_with_snapshot(model, sorter, config_diffusion):

    num_blocks = config_diffusion.num_blocks
    step_per_block = config_diffusion.step_per_block
    size_block = config_diffusion.size_block
    id_mask = config_diffusion.id_mask
    len_prompt = config_diffusion.len_prompt

    idx_refresh = torch.tensor([[]])

    for id_block in range(num_blocks):
        position_start = len_prompt + id_block * size_block
        position_end = position_start + size_block
        mask_mask_blk = x[:,:position_end] == id_mask

        quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)


        idx_denoising = x[:, position_start:position_end]
        x_current = torch.cat([idx_refresh, idx_denoising], dim=-1) 
        logits = model(x_current, idx_refresh=idx_refresh, idx_denoising=idx_denoising)

        snapshot = SimpleLogitsSnapshot(logits, x, y)\
                    .set_mask_id(id_mask)\
                    .transform_logits(mask_mask_blk)
        # end

        for step in range(step_per_block):
            idx_sorted_by_conf = snapshot.generate_sorted_sequence(sorter)
            num_unmask = quota_helper.get_quota(step)
            idx_unmask = idx_sorted_by_conf[:, :num_unmask]

            if step == 0:
                snapshot.unmask_by_idx(idx_unmask)  # get the x0, x0_p
                idx_refresh = idx_unmask
                continue
            # end

            idx_denoising = idx_unmask
            idx_current = torch.cat([idx_refresh, idx_denoising], dim=-1)   # 原本的位置
            x_current = x[:, idx_current]
            logits = model(x_current, idx_current=idx_current) # 有问题兄弟

            logits_refresh = logits[:,:idx_refresh.shape[-1]]
            logits_denoising = logits[:, idx_denoising.shape[-1]:]

            snapshot.update_logits(logits_refresh, idx_refresh)
            snapshot.unmask_by_logits(logits_denoising, idx_denoising)

            idx_refresh = idx_denoising
        # end


    # end
# end